# PDF and OCR extraction evaluation

Only fictional, labelled invoices are used. This notebook runs the actual local extractor and compares 12 fields per document with independently generated ground truth. It measures field extraction, not real-world OCR reliability or payment approval.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd() if (Path.cwd() / "reconciliation").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from reconciliation.evaluation import evaluate

documents, fields, summary = evaluate(ROOT / "data")

## Digital PDF: actual text-layer result

In [2]:
digital = next(d for d in documents if d["filename"] == "invoice_a_digital.pdf")
print("Method:", digital["method"])
print(digital["text"])
print("Errors:", digital["errors"])

Method: digital
SYNTHETIC DEMO — NOT A REAL INVOICE.

Fictional equipment rental | One line item | USD

Invoice number: DEMO-A

Vendor: Example Equipment Rentals

PO number: PO-1001

Equipment ID: EX-01

Equipment name: Excavator

Quantity: 1

Rental days: 3

Rental start: 2026-11-02

Rental end: 2026-11-04

Rate basis: daily

Daily rate: 450.00

Line total: 1350.00





Rental dates are inclusive. No tax, fees, or customer billing in this demo.
SYNTHETIC DEMO — NOT A REAL INVOICE.
Errors: []


## Image-only PDF: actual Tesseract result

The fixture contains pixels only; there is no hidden text layer.

In [3]:
import pymupdf
import pytesseract

with pymupdf.open(ROOT / "data/invoices/invoice_b_scanned.pdf") as pdf:
    assert not pdf[0].get_text().strip()
scan = next(d for d in documents if d["filename"] == "invoice_b_scanned.pdf")
print("Tesseract:", pytesseract.get_tesseract_version())
print("Method:", scan["method"])
print(scan["text"])
print("Warnings:", scan["warnings"])
print("Errors:", scan["errors"])

Tesseract: 5.5.0
Method: ocr
SYNTHETIC DEMO — NOT A REAL INVOICE.
Fictional equipment rental | One line item | USD
Invoice number: DEMO-B

Vendor: Sample Lift Services

PO number: PO-1002

Equipment ID: SL-01

Equipment name: Scissor Lift

Quantity: 1

Rental days: 2

Rental start: 2026-11-05

Rental end: 2026-11-06

Rate basis: daily

Daily rate: 150.00

Line total: 300.00

Rental dates are inclusive. No tax, fees, or customer billing in this demo.
SYNTHETIC DEMO — NOT A REAL INVOICE.

Warnings: ['Page 1: OCR text must be checked against the image; table layout is not guaranteed.']
Errors: []


## Ground-truth field comparisons

Money is compared numerically as Decimal. Other fields must exactly equal the expected value; absence must remain absent. Digital includes five documents; scanned includes one.

In [4]:
fields

,filename,method,field,expected,actual,correct
0,invoice_a_digital.pdf,digital,invoice_number,DEMO-A,DEMO-A,True
1,invoice_a_digital.pdf,digital,vendor,Example Equipment Rentals,Example Equipment Rentals,True
2,invoice_a_digital.pdf,digital,po_number,PO-1001,PO-1001,True
3,invoice_a_digital.pdf,digital,equipment_id,EX-01,EX-01,True
4,invoice_a_digital.pdf,digital,equipment_name,Excavator,Excavator,True
...,...,...,...,...,...,...
67,invoice_f_wrong_dates.pdf,digital,rental_start,2026-11-07,2026-11-07,True
68,invoice_f_wrong_dates.pdf,digital,rental_end,2026-11-08,2026-11-08,True
69,invoice_f_wrong_dates.pdf,digital,rate_basis,daily,daily,True
70,invoice_f_wrong_dates.pdf,digital,daily_rate,150.00,150.00,True


In [5]:
summary

,method,correct,total,accuracy
0,digital,60,60,1.0
1,ocr,12,12,1.0


In [6]:
errors = fields.loc[~fields["correct"]]
print(f"Field errors: {len(errors)}")
errors

Field errors: 0


,filename,method,field,expected,actual,correct


## Limitations and next experiments

These measurements cover six generated, single-line invoices, including one clean 220-DPI scan. They do not establish accuracy on unseen vendors, skewed or noisy scans, handwriting, tables, multiple line items, taxes, or credits. The parser recognizes explicit English labels and ISO dates only; it is deterministic and uses no LLM. Text-layer selection uses a simple character-count heuristic.

Next: obtain permission for a small representative, de-identified sample; manually label a held-out set; compare native extraction and OCR by document type, field, error severity, and runtime. Test rotation, low contrast, and multi-line layouts before considering a local model. Do not tune and report accuracy on the same documents. Business owners must define tolerances and review rules before any operational use.